In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("combined").getOrCreate()
sc = spark.sparkContext
print("created")

created


In [7]:
num = [10,20,30,40,50]
rdd = sc.parallelize(num)
rdd.collect()

[10, 20, 30, 40, 50]

In [8]:
rdd.getNumPartitions()

2

In [9]:
rdd.glom().collect()

[[10, 20], [30, 40, 50]]

In [12]:
mapped = rdd.map(lambda x:x * 30)
mapped.collect()

[300, 600, 900, 1200, 1500]

In [13]:
rdd = sc.parallelize(["viswa", "siva"])
ma = rdd.map(lambda x : x.upper())
ma.collect()

['VISWA', 'SIVA']

In [14]:
fi = rdd.filter(lambda x: len(x)>4)
fi.collect()

['viswa']

1. map() → Transform each element
👉 What it does:
Applies a function to every row
Returns a new transformed value for each row
Output size = same as input size

filter() → Select specific elements
👉 What it does:
Applies a condition
Keeps only elements that satisfy the condition
Output size = less than or equal to input
📌 Example:

In [15]:
lines = sc.parallelize(["spark and hadoop", "has a top"])
fm = lines.flatMap(lambda x : x.split(" "))
fm.collect()

['spark', 'and', 'hadoop', 'has', 'a', 'top']

In [16]:
rdd1 = sc.parallelize([1,2,3,4])
rdd2 = sc.parallelize([3,4])
rdd1.union(rdd2).collect()

[1, 2, 3, 4, 3, 4]

In [17]:
rdd1.intersection(rdd2).collect()

[4, 3]

In [18]:
rdd1.subtract(rdd2).collect()

[1, 2]

**actions**

In [19]:
rdd1.count()

4

In [20]:
rdd1.first()

1

In [22]:
rdd1.take(3)

[1, 2, 3]

In [33]:
data = [
    ("IT", 7000),
    ("HR", 6000),
    ("Finance", 8000),
    ("IT", 500)
]
rdd = sc.parallelize(data)
rdd.collect()

[('IT', 7000), ('HR', 6000), ('Finance', 8000), ('IT', 500)]

In [34]:
rdd.reduceByKey(lambda a, b: a + b).collect()

[('IT', 7500), ('HR', 6000), ('Finance', 8000)]

In [35]:
result = rdd.groupByKey()

result.mapValues(list).collect()

[('IT', [7000, 500]), ('HR', [6000]), ('Finance', [8000])]

In [36]:
rdd.sortBy(lambda x:x).collect()

[('Finance', 8000), ('HR', 6000), ('IT', 500), ('IT', 7000)]

Good — now you’re asking the **right deep question** 👍

---

## 🔥 On what basis does sorting happen here?

You used:

```python
rdd.sortBy(lambda x: x)
```

👉 So sorting is based on:

> **the entire tuple → (key, value)**

---

## 🧠 Actual Rule Used → **Lexicographical Ordering**

For tuples:

```python
("HR", 6000)
("IT", 7000)
("finance", 8000)
```

Sorting happens like this:

---

## 🔹 Step 1: Compare first element (key)

👉 Spark compares:

```python
"HR" vs "IT" vs "finance"
```

### Based on:

👉 **ASCII / Unicode order of characters**

---

## 🔥 Important: ASCII Ordering

| Character | ASCII Value |
| --------- | ----------- |
| H         | 72          |
| I         | 73          |
| f         | 102         |

👉 So:

```python
"H" < "I" < "f"
```

---

## ✅ Final Order

```python
("HR", 6000)
("IT", 7000)
("finance", 8000)
```

✔ Sorted based on **first element only**

---

## 🔹 Step 2: When second element is used?

👉 Only if first elements are SAME

### Example:

```python
("IT", 7000)
("IT", 3000)
```

Now comparison becomes:

```python
7000 vs 3000
```

👉 Result:

```python
("IT", 3000)
("IT", 7000)
```

---

## 🔥 Summary Rule

👉 Sorting happens like:

1. Compare **x[0] (key)**
2. If equal → compare **x[1] (value)**

---

## 🧠 Simple Understanding

* First priority → **department name**
* Second priority → **salary (only if same department)**

---

## ⚠️ Important Insight (Interview Gold)

👉 Sorting is:

* **Case-sensitive**
* Uppercase comes before lowercase

So:

```python
"IT" < "finance"
```

---

## 🚀 Final Answer (Interview Style)

👉 *“When using sortBy(lambda x: x), PySpark sorts tuples using lexicographical ordering—first comparing the key, and if keys are equal, comparing the value.”*

---

If you want next:
👉 I can show **how sorting behaves with mixed cases + numbers (tricky edge cases)** 😄


In [32]:
lines = sc.parallelize([
    "hello world",
    "hello spark",
    "hello world spark"
])

word_count = (
    lines
    .flatMap(lambda line: line.split(" "))   # split into words
    .map(lambda word: (word, 1))             # (word, 1)
    .reduceByKey(lambda a, b: a + b)         # sum counts
)

print(word_count.collect())

[('hello', 3), ('world', 2), ('spark', 2)]
